# 0. import libraries

In [2]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [1]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Base")

model = ReplacementModel.from_pretrained(
    "Qwen/Qwen3-4B-Base",
    "mwhanna/qwen3-4b-transcoders",
    dtype=torch.bfloat16,
    backend="nnsight",
    lazy_encoder=True,
    lazy_decoder=True,
)

Fetching 36 files:   0%|          | 0/36 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

# 1. getting attributes from each token step
based on attribute_demo  
**can skip to step 2, features are saved in** `demos\graph_files`

In [3]:
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 1024  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 32  # Batch size when attributing
offload = "cpu"
verbose = True  # Whether to display a tqdm progress bar and timing report

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [5]:
# ── 3. Generation loop (same as before, but using ReplacementModel) ───────────
base_prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"].to(model.device)
STOP_IDS = {198, 151645, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out.logits[0, -1, :].float()
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
            [input_ids, torch.tensor([[next_id]], device=input_ids.device)], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        max_feature_nodes=max_feature_nodes,   # ← missing
        batch_size=batch_size,
        offload=offload,                        # ← use the variable, not hardcoded "cpu"
        verbose=verbose,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files/qwen3-4b/",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

Phase 0: Precomputing activations and vectors


Generated: But the rabbit was too fast and he couldn't catch it.


Precomputation completed in 22.42s
Found 61084 active features
Phase 1: Running forward pass
Forward pass completed in 0.34s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5391
Will include 1024 of 61084 feature nodes
Input vectors built in 0.19s
Phase 3: Computing logit attributions
1 logit attribution(s) completed in 2.26s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:08<00:00, 125.91it/s]
Feature attributions completed in 8.14s
Attribution completed in 34.41s
Phase 0: Precomputing activations and vectors


✓ step 00 → 'But'  saved as 'step-00-But'


Precomputation completed in 23.37s
Found 62937 active features
Phase 1: Running forward pass
Forward pass completed in 0.17s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.6328
Will include 1024 of 62937 feature nodes
Input vectors built in 0.18s
Phase 3: Computing logit attributions
1 logit attribution(s) completed in 2.19s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:08<00:00, 122.21it/s]
Feature attributions completed in 8.38s
Attribution completed in 35.27s
Phase 0: Precomputing activations and vectors


...

✓ step 10 → ' catch'  saved as 'step-10-catch'


Precomputation completed in 23.75s
Found 78351 active features
Phase 1: Running forward pass
Forward pass completed in 0.36s
Phase 2: Building input vectors
Using 1 salient logits with cumulative probability 0.9805
Will include 1024 of 78351 feature nodes
Input vectors built in 0.07s
Phase 3: Computing logit attributions
1 logit attribution(s) completed in 2.67s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:09<00:00, 105.26it/s]
Feature attributions completed in 9.73s
Attribution completed in 37.09s
Phase 0: Precomputing activations and vectors


✓ step 11 → ' it'  saved as 'step-11-it'


Precomputation completed in 24.49s
Found 83101 active features
Phase 1: Running forward pass
Forward pass completed in 0.45s
Phase 2: Building input vectors
Using 7 salient logits with cumulative probability 0.9570
Will include 1024 of 83101 feature nodes
Input vectors built in 0.08s
Phase 3: Computing logit attributions
1 logit attribution(s) completed in 3.96s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [01:06<00:00, 15.46it/s]
Feature attributions completed in 66.24s
Attribution completed in 95.66s


✓ step 12 → '.'  saved as 'step-12-.'


In [10]:
# extracting clerp descriptions and adding to graph files

import json, time, requests
from pathlib import Path

def fetch_clerp(layer, feature):
    url = f"https://www.neuronpedia.org/api/feature/qwen3-4b/{layer}-transcoder-hp/{feature}"
    r = requests.get(url)
    if r.ok:
        data = r.json()
        descs = data.get("explanations", [])
        if descs:
            return descs[0].get("description", "")
    return ""

for json_path in sorted(Path("./graph_files/qwen3-4b/").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    changed = False
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        if node.get('clerp'):  # skip if already fetched
            continue
        parts = node['node_id'].split('_')
        feat = parts[1]
        layer = node['layer']
        node['clerp'] = fetch_clerp(layer, feat)
        changed = True
        time.sleep(0.1)
    if changed:
        json_path.write_text(json.dumps(data))
    print(f"done: {json_path.name}")

done: step-00-But.json


KeyboardInterrupt: 

# 2. attribute graph visualization

In [1]:
import json
from pathlib import Path

graph_dir = Path("./graph_files/qwen3-4b")

for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    
    # collect all unique layers used by transcoder nodes
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    
    transcoder_list = [f"qwen3-4b/{l}-gemmascope-transcoder-16k" for l in layers]
    data["metadata"]["transcoder_list"] = transcoder_list
    
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: {len(layers)} layers → {transcoder_list[:3]}...")

# also patch the manifest
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest patched")

step-00-But.json: 36 layers → ['qwen3-4b/0-gemmascope-transcoder-16k', 'qwen3-4b/1-gemmascope-transcoder-16k', 'qwen3-4b/2-gemmascope-transcoder-16k']...
step-01-the.json: 36 layers → ['qwen3-4b/0-gemmascope-transcoder-16k', 'qwen3-4b/1-gemmascope-transcoder-16k', 'qwen3-4b/2-gemmascope-transcoder-16k']...
step-02-rabbit.json: 36 layers → ['qwen3-4b/0-gemmascope-transcoder-16k', 'qwen3-4b/1-gemmascope-transcoder-16k', 'qwen3-4b/2-gemmascope-transcoder-16k']...
step-03-was.json: 35 layers → ['qwen3-4b/0-gemmascope-transcoder-16k', 'qwen3-4b/1-gemmascope-transcoder-16k', 'qwen3-4b/2-gemmascope-transcoder-16k']...
step-04-too.json: 36 layers → ['qwen3-4b/0-gemmascope-transcoder-16k', 'qwen3-4b/1-gemmascope-transcoder-16k', 'qwen3-4b/2-gemmascope-transcoder-16k']...
step-05-fast.json: 36 layers → ['qwen3-4b/0-gemmascope-transcoder-16k', 'qwen3-4b/1-gemmascope-transcoder-16k', 'qwen3-4b/2-gemmascope-transcoder-16k']...
step-06-and.json: 36 layers → ['qwen3-4b/0-gemmascope-transcoder-16k', '

In [2]:
import json
from pathlib import Path

data = {"graphs": []}  # ← Add this line to initialize

for json_path in sorted(Path("./graph_files/qwen3-4b").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/qwen3-4b/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


In [3]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/qwen3-4b/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

http://localhost:8046/index.html


## 2.1 feature analysis based on circuit

In [11]:
import json
from pathlib import Path
from collections import defaultdict

# collect top 10 nodes per step
top_features = set()

for json_path in sorted(Path("./graph_files/qwen3-4b/").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    nodes = [n for n in data['nodes'] if not n['is_target_logit'] and n['influence'] is not None]
    nodes = sorted(nodes, key=lambda x: x['influence'], reverse=True)[:10]
    for node in nodes:
        parts = node['node_id'].split('_')
        top_features.add((node['layer'], parts[1]))

print(f"Unique top features across all steps: {len(top_features)}")

Unique top features across all steps: 102


In [13]:
data = json.loads(Path("./graph_files/qwen3-4b/step-00-But.json").read_text())
layers = set(n['layer'] for n in data['nodes'])
print(sorted(layers))

['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '37', '4', '5', '6', '7', '8', '9', 'E']


In [12]:
import requests, time

def fetch_clerp(layer, feature):
    url = f"https://www.neuronpedia.org/api/feature/qwen3-4b/{layer}-transcoder-hp/{feature}"
    r = requests.get(url)
    if r.ok:
        descs = r.json().get("explanations", [])
        if descs:
            return descs[0].get("description", "")
    return ""

clerp_cache = {}
for i, (layer, feat) in enumerate(top_features):
    key = f"L{layer}F{feat}"
    clerp_cache[key] = fetch_clerp(layer, feat)
    time.sleep(0.1)
    if i % 10 == 0:
        print(f"{i}/{len(top_features)} fetched...")

print("done")

0/102 fetched...
10/102 fetched...
20/102 fetched...
30/102 fetched...
40/102 fetched...
50/102 fetched...
60/102 fetched...
70/102 fetched...
80/102 fetched...
90/102 fetched...
100/102 fetched...
done


In [14]:
for key, clerp in sorted(clerp_cache.items()):
    if clerp:  # only show features that have descriptions
        print(f"{key}: {clerp}")

L0F100727: quotations and excerpts
L0F104841: Code and file references
L0F116738: heralded, promoted
L0F133923: code
L0F56041: had
L0F74566: Code/non-English text
L10F10: Crystal Palace, Mount Vernon
L10F110470: _of
L10F139856: catch
L10F153729: ot
L10F51359: y
L11F11: conjunctions
L11F122209: I
L11F135338: punctuation
L12F15526: embeddings
L12F20200: code examples
L12F93427: stories
L13F13: body, breathing
L13F3127: Code, text, and writing
L13F67631: too
L14F44184: Negation
L14F78631: He
L15F15: certifications and best regards
L15F17068: song lyrics and poetry
L17F105400: Scraped/artificial text
L17F143074: Code and licensing text
L17F96218: Song lyrics/poems
L18F142989: Song lyrics/poems
L18F22391: mixed documents/files
L19F126960: my
L1F1: ws
L1F105903: a
L1F156989: Names
L22F129873: period
L23F91721: Technology and science texts
L28F58760: the
L28F67071: non-English or code
L29F155625: technical writing/forum posts
L29F18221: unfortunately, sadly
L29F85632: and
L2F19661: the
L2F576

In [15]:
# find which steps each poetry feature appears in
poetry_features = {k for k, v in clerp_cache.items() if v and any(
    word in v.lower() for word in ['poem', 'poet', 'lyric', 'rhym', 'song', 'verse']
)}

print("Poetry-related features:", poetry_features)
print()

for json_path in sorted(Path("./graph_files/qwen3-4b/").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    slug = data['metadata']['slug']
    
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        parts = node['node_id'].split('_')
        key = f"L{node['layer']}F{parts[1]}"
        if key in poetry_features:
            print(f"  {slug}  {key}  influence={node['influence']:.4f}  {clerp_cache[key]}")

Poetry-related features: {'L9F6500', 'L17F96218', 'L15F17068', 'L18F142989', 'L5F81584', 'L4F144104'}

  step-00-But  L5F81584  influence=0.7108  Rhythm or rhyming
  step-00-But  L9F6500  influence=0.7577  Code/Verse/Scientific-adjacent
  step-00-But  L9F6500  influence=0.7367  Code/Verse/Scientific-adjacent
  step-00-But  L17F96218  influence=0.6494  Song lyrics/poems
  step-01-the  L4F144104  influence=0.7977  poetry and song lyrics
  step-01-the  L5F81584  influence=0.7523  Rhythm or rhyming
  step-01-the  L9F6500  influence=0.7432  Code/Verse/Scientific-adjacent
  step-01-the  L17F96218  influence=0.7803  Song lyrics/poems
  step-01-the  L18F142989  influence=0.7915  Song lyrics/poems
  step-02-rabbit  L4F144104  influence=0.6985  poetry and song lyrics
  step-02-rabbit  L5F81584  influence=0.7881  Rhythm or rhyming
  step-02-rabbit  L17F96218  influence=0.7674  Song lyrics/poems
  step-03-was  L4F144104  influence=0.7737  poetry and song lyrics
  step-03-was  L5F81584  influence=0

**Rhyme/poetry features:**
- `L4F144104: poetry and song lyrics`
- `L5F81584: Rhythm or rhyming`
- `L15F17068: song lyrics and poetry`
- `L17F96218: Song lyrics/poems`
- `L18F142989: Song lyrics/poems`

**Semantically relevant to the prompt:**
- `L6F97909: carrot/carrots` — directly prompt-specific
- `L8F44236: carrot, center, carroll` — also carrot-related
- `L2F57682: saw` — from "He saw a carrot"
- `L3F53796: grab` — from "had to grab it"
- `L14F78631: He` — subject of the couplet
- `L7F10667: saw`

**Potentially interesting for the rhyming mechanism:**
- `L13F67631: too` — the model generated "too" at step 04
- `L11F11: conjunctions`
- `L29F18221: unfortunately, sadly` — negative sentiment

The most interesting finding for your replication is that you have **multiple dedicated poetry/rhyming features** (`L4F144104`, `L5F81584`, `L15F17068`, `L17F96218`, `L18F142989`) active in the top influential nodes. That's directly analogous to what the original Gemma experiment found with `L14F15021`.

# 3. intervention
based on intervention_demo

In [4]:
from collections import namedtuple
from functools import partial

# display functions
from circuit_tracer.utils.demo_utils import display_topk_token_predictions, display_generations_comparison

In [5]:
Feature = namedtuple('Feature', ['layer', 'pos', 'feature_idx'])

# a display function that needs the model's tokenizer
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

In [4]:
supernode_features = [
    Feature(layer=4, pos=-1, feature_idx=144104),
    Feature(layer=5, pos=-1, feature_idx=81584),
    Feature(layer=15, pos=-1, feature_idx=17068),
    Feature(layer=17, pos=-1, feature_idx=96218),
    Feature(layer=18, pos=-1, feature_idx=142989),
]

intervention_tuples = [(*supernode_feature, 0.0) for supernode_feature in supernode_features]

In [2]:
s = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

In [7]:
with torch.no_grad():
    original_logits, _ = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

Token,Probability,Distribution
He,0.145,14.5%
But,0.112,11.2%
And,0.053,5.3%
It,0.042,4.2%
The,0.042,4.2%
Token,Probability,Distribution
He,0.144,14.4%
But,0.112,11.2%
And,0.053,5.3%
It,0.041,4.1%


In [6]:
from circuit_tracer.utils.demo_utils import display_generations_comparison

# suppress both candidate features at the newline position (-1)
intervention_tuples = [
    (4, -1, 144104, 0.0),
    (5, -1, 81584, 0.0),
    (15, -1, 17068, 0.0),
    (17, -1, 96218, 0.0),
    (18, -1, 142989, 0.0),
]

print("Generating with original features...")
pre = [model.feature_intervention_generate(s, [], do_sample=False, verbose=True)[0]]

print("\nGenerating with interventions...")
post = [model.feature_intervention_generate(s, intervention_tuples, do_sample=False, verbose=True)[0]]

display_generations_comparison(s, pre, post)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generating with original features...


c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\nnsight\intervention\interleaver.py:422: UserWarning: Execution complete but `model.model.layers.0.post_attention_layernorm.output.i11` was not provided. If this was in an Iterator at iteration None this iteration did not happen. If you were using `.iter[:]`, this is likely not an error.
  warnings.warn(msg)
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\nnsight\intervention\interleaver.py:422: UserWarning: Execution complete but `model.model.layers.0.mlp.output.i11` was not provided. If this was in an Iterator at iteration None this iteration did not happen. If you were using `.iter[:]`, this is likely not an error.
  warnings.warn(msg)



Generating with interventions...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\nnsight\intervention\interleaver.py:422: UserWarning: Execution complete but `model.model.layers.0.post_attention_layernorm.output.i32` was not provided. If this was in an Iterator at iteration None this iteration did not happen. If you were using `.iter[:]`, this is likely not an error.
  warnings.warn(msg)
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\nnsight\intervention\interleaver.py:422: UserWarning: Execution complete but `model.model.layers.0.mlp.output.i32` was not provided. If this was in an Iterator at iteration None this iteration did not happen. If you were using `.iter[:]`, this is likely not an error.
  warnings.warn(msg)


In [12]:
s = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

with torch.no_grad():
    original_logits, _ = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

Token,Probability,Distribution
He,0.131,13.1%
But,0.131,13.1%
And,0.070,7.0%
The,0.042,4.2%
Then,0.037,3.7%
Token,Probability,Distribution
He,0.134,13.4%
But,0.118,11.8%
And,0.072,7.2%
The,0.043,4.3%


In [14]:
intervention_tuples = []
for layer, feat in [(4, 144104), (5, 81584), (15, 17068), (17, 96218), (18, 142989)]:
    for pos in range(-10, 0):  # suppress at last 10 positions
        intervention_tuples.append((layer, pos, feat, 0.0))

with torch.no_grad():
    original_logits, _ = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

Token,Probability,Distribution
He,0.131,13.1%
But,0.131,13.1%
And,0.070,7.0%
The,0.042,4.2%
Then,0.037,3.7%
Token,Probability,Distribution
He,0.116,11.6%
But,0.116,11.6%
And,0.070,7.0%
Then,0.043,4.3%
